# Урок 19 — Реальний Кейс: Потокова Обробка Біржових Транзакцій

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

> **Сценарій:** Ви — старший інженер даних у брокерській компанії.  
> Ваша система щодня отримує мільйони рядків із біржовими угодами українських компаній.  
> Ваше завдання — побудувати **стрімінговий конвеєр**, який обробляє ці дані в реальному часі,  
> не завантажуючи весь файл у пам'ять.

---

## 0. HOOK: Проблема 5-Гігабайтного Файлу

---

Уявіть: ваш колега пише такий код:

```python
with open('transactions.ndjson') as f:
    data = f.readlines()   # 💥 5 GB у RAM одразу
```

О 3:00 ночі сервер падає з `MemoryError`. Клієнти не можуть торгувати. Збитки — мільйони.

**Де помилка?** Не в алгоритмі. В моделі мислення.  
Колега думав про дані як про **склад (warehouse)**: спочатку завезти все, потім обробляти.  
Ми будемо мислити про дані як про **конвеєр (pipeline)**: обробляти по одній деталі за раз.

| Підхід | Аналогія | RAM для 5 ГБ |
|--------|----------|-------------|
| `list` / `readlines()` | Завезти всю сировину на склад | ~5 ГБ |
| Generator / Streaming | Конвеєр: деталь → обробка → виріб | ~кілька КБ |

---

## 1. Підготовка: Налаштування середовища та конфігурація

---

In [ ]:
import json
import csv
import random
import sys
import os
from datetime import datetime, timedelta
from pathlib import Path
from itertools import islice, takewhile, chain
from functools import reduce

random.seed(42)

COMPANIES = {
    "Нафтогаз":  {"base_price": 145.0, "volatility": 0.015},
    "ПриватБанк": {"base_price":  89.0, "volatility": 0.010},
    "Розетка":   {"base_price": 234.0, "volatility": 0.020},
    "Укрнафта":  {"base_price":  67.0, "volatility": 0.012},
    "Київстар":  {"base_price": 312.0, "volatility": 0.008},
}

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

JSON_FILE = DATA_DIR / "transactions.ndjson"
CSV_FILE  = DATA_DIR / "transactions.csv"

print("Конфігурація завантажена.")
print(f"Компанії: {', '.join(COMPANIES)}")

## 2. Ментальна Модель: List vs Iterator vs Generator

---

Перш ніж писати систему, вирівняємо ментальні моделі.

```
LIST        → фотографія даних      → все в пам'яті одразу
ITERATOR    → касета з плівкою      → кадр видається за запитом
GENERATOR   → режисер на знімальному майданчику → кадр знімається лише коли потрібен
```

Всі три дозволяють писати `for item in obj:`, але їх поведінка кардинально різна.

In [ ]:
N = 1_000_000

eager = [i for i in range(N)]
lazy  = (i for i in range(N))

print(f"List:      {sys.getsizeof(eager):>12,} байт  ({sys.getsizeof(eager)/1024/1024:.1f} МБ)")
print(f"Generator: {sys.getsizeof(lazy):>12,} байт")

## 3. Як Python Виконує `for` — Внутрішня Механіка

---

Цикл `for` — це лише синтаксичний цукор над двома функціями: `iter()` і `next()`.  
Розуміння цього механізму — це ключ до розуміння всієї системи потоків даних.

In [ ]:
companies = ["Нафтогаз", "ПриватБанк", "Розетка"]

print("=== Те, що ви пишете ===")
for company in companies:
    print(f"  → {company}")

print("\n=== Те, що виконує Python ===")
it = iter(companies)
print(f"iter() повернув: {type(it)}")
while True:
    try:
        company = next(it)
        print(f"  → {company}")
    except StopIteration:
        print("  ⏹ StopIteration: потік вичерпано")
        break

## 4. Власний Ітератор: Клас TransactionIterator

---

Реалізуємо протокол ітератора вручну, щоб побачити, як він влаштований зсередини.  

Наш клас буде генерувати одну біржову транзакцію за викликом `next()`,  
зберігаючи поточну ціну акцій у локальному стані між викликами.

In [ ]:
class TransactionIterator:
    def __init__(self, company: str, base_price: float, volatility: float, count: int):
        self.company = company
        self.price = base_price
        self.volatility = volatility
        self.count = count
        self._generated = 0
        self._ts = datetime(2024, 1, 1, 9, 0, 0)
        self._trade_id = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self._generated >= self.count:
            raise StopIteration
        self.price *= (1 + random.gauss(0, self.volatility))
        self.price = max(self.price, 1.0)
        self._ts += timedelta(seconds=random.randint(1, 30))
        tx = {
            "id":        self._trade_id,
            "company":   self.company,
            "price":     round(self.price, 2),
            "volume":    random.randint(100, 10_000),
            "timestamp": self._ts.isoformat(),
        }
        self._generated += 1
        self._trade_id += 1
        return tx


it = TransactionIterator("Нафтогаз", base_price=145.0, volatility=0.015, count=3)
for tx in it:
    print(tx)

## 5. Generator Pattern: `yield` — Та сама логіка, вдвічі менше коду

---

Клас `TransactionIterator` вимагає `__init__`, `__iter__`, `__next__` та ручного управління станом.  
Генератор реалізує ту ж саму поведінку — `yield` автоматично зберігає весь стан функції між викликами.

In [ ]:
def transaction_generator(company: str, base_price: float, volatility: float, count: int = None):
    price = base_price
    ts = datetime(2024, 1, 1, 9, 0, 0)
    trade_id = 1
    generated = 0

    while count is None or generated < count:
        price *= (1 + random.gauss(0, volatility))
        price = max(price, 1.0)
        ts += timedelta(seconds=random.randint(1, 30))
        yield {
            "id":        trade_id,
            "company":   company,
            "price":     round(price, 2),
            "volume":    random.randint(100, 10_000),
            "timestamp": ts.isoformat(),
        }
        generated += 1
        trade_id += 1


gen = transaction_generator("ПриватБанк", base_price=89.0, volatility=0.010, count=3)
for tx in gen:
    print(tx)

## 6. Генерація Даних: Запис NDJSON та CSV файлів

---

Будуємо генератор `all_companies_stream`, який об'єднує потоки від усіх компаній  
через `itertools.chain` і записує дані в два формати:

- **NDJSON** (Newline-Delimited JSON) — кожна транзакція = один рядок JSON.
- **CSV** — класичний табличний формат.

Ключовий момент: файл записується **рядок за рядком**, не буферизуючи весь датасет у пам'яті.

In [ ]:
def all_companies_stream(companies: dict, count_per_company: int):
    streams = [
        transaction_generator(name, cfg["base_price"], cfg["volatility"], count_per_company)
        for name, cfg in companies.items()
    ]
    yield from chain(*streams)


def write_ndjson(path: Path, stream, total: int) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for tx in stream:
            f.write(json.dumps(tx, ensure_ascii=False) + "\n")
    print(f"  ✓ NDJSON записано: {path} ({total} транзакцій)")


def write_csv(path: Path, stream, total: int) -> None:
    fields = ["id", "company", "price", "volume", "timestamp"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for tx in stream:
            writer.writerow(tx)
    print(f"  ✓ CSV  записано: {path} ({total} транзакцій)")


COUNT = 2_000

random.seed(42)
write_ndjson(JSON_FILE, all_companies_stream(COMPANIES, COUNT), COUNT * len(COMPANIES))

random.seed(42)
write_csv(CSV_FILE, all_companies_stream(COMPANIES, COUNT), COUNT * len(COMPANIES))

print(f"\nРозмір NDJSON: {JSON_FILE.stat().st_size / 1024:.1f} КБ")
print(f"Розмір CSV:    {CSV_FILE.stat().st_size / 1024:.1f} КБ")

## 7. Stream Reading: Читання Файлів без Завантаження в RAM

---

Тепер реалізуємо читачів — генератори, які відкривають файл і видають транзакції по одній.  
Зверніть увагу: `for line in file` — це вже протокол ітератора. Файловий об'єкт у Python є ітератором.

In [ ]:
def read_ndjson(path: Path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)


def read_csv(path: Path):
    with open(path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            yield {
                "id":        int(row["id"]),
                "company":   row["company"],
                "price":     float(row["price"]),
                "volume":    int(row["volume"]),
                "timestamp": row["timestamp"],
            }


stream = read_ndjson(JSON_FILE)
print("Перші 3 транзакції з NDJSON (лінива читалка):")
for tx in islice(stream, 3):
    print(f"  [{tx['id']:>4}] {tx['company']:<12} ₴{tx['price']:>8.2f}  vol={tx['volume']}")

## 8. Pipeline: Конвеєр Трансформацій

---

Будуємо Data Pipeline з трьох незалежних генераторів.
Кожен крок ізольований, тестований, і не знає нічого про сусідів.

```
read_ndjson() → filter_by_price() → extract_value() → aggregate()
     ↓                 ↓                  ↓                ↓
  Source            Filter              Map              Sink
```

In [ ]:
def filter_by_company(stream, company: str):
    for tx in stream:
        if tx["company"] == company:
            yield tx


def filter_by_price(stream, min_price: float = 0, max_price: float = float("inf")):
    for tx in stream:
        if min_price <= tx["price"] <= max_price:
            yield tx


def enrich_with_total(stream):
    for tx in stream:
        yield {**tx, "total": round(tx["price"] * tx["volume"], 2)}


def aggregate(stream):
    count = total_value = total_volume = 0
    min_price = float("inf")
    max_price = 0.0
    for tx in stream:
        total_value  += tx["price"] * tx["volume"]
        total_volume += tx["volume"]
        min_price = min(min_price, tx["price"])
        max_price = max(max_price, tx["price"])
        count += 1
    return {
        "count":       count,
        "total_value": round(total_value, 2),
        "avg_price":   round(total_value / total_volume, 2) if total_volume else 0,
        "min_price":   round(min_price, 2),
        "max_price":   round(max_price, 2),
        "vwap":        round(total_value / total_volume, 2) if total_volume else 0,
    }

In [ ]:
target = "Розетка"

pipeline = enrich_with_total(
    filter_by_price(
        filter_by_company(
            read_ndjson(JSON_FILE),
            company=target,
        ),
        min_price=200.0,
    )
)

stats = aggregate(pipeline)

print(f"📊 Аналіз транзакцій: {target} (ціна > 200 ₴)")
print(f"   Кількість угод:  {stats['count']:>8,}")
print(f"   Загальний обсяг: {stats['total_value']:>14,.2f} ₴")
print(f"   Ціна мін/макс:   {stats['min_price']:>8.2f} / {stats['max_price']:.2f} ₴")
print(f"   VWAP (зважена):  {stats['vwap']:>8.2f} ₴")

## 9. Аналіз Усіх Компаній: Ітерація по Потоку

---

Тепер розрахуємо VWAP (Volume-Weighted Average Price) для кожної компанії  
за один прохід через файл, без завантаження в пам'ять.

In [ ]:
from collections import defaultdict

def vwap_by_company(path: Path) -> dict:
    totals  = defaultdict(float)
    volumes = defaultdict(int)
    counts  = defaultdict(int)

    for tx in read_ndjson(path):
        c = tx["company"]
        totals[c]  += tx["price"] * tx["volume"]
        volumes[c] += tx["volume"]
        counts[c]  += 1

    return {
        c: {
            "vwap":   round(totals[c] / volumes[c], 2),
            "trades": counts[c],
            "volume": volumes[c],
        }
        for c in totals
    }


results = vwap_by_company(JSON_FILE)

print(f"{'Компанія':<14} {'VWAP':>10} {'Угод':>8} {'Обсяг':>12}")
print("-" * 48)
for company, stats in sorted(results.items(), key=lambda x: -x[1]["vwap"]):
    print(f"{company:<14} {stats['vwap']:>9.2f}₴ {stats['trades']:>8,} {stats['volume']:>12,}")

## 10. Вбудовані Інструменти: `map`, `filter`, `zip`, `enumerate`, `any`/`all`

---

Python має вбудовані функціональні інструменти, які є ледачими ітераторами і ідеально  
вписуються в конвеєри без зайвого копіювання в пам'ять.

In [ ]:
sample = list(islice(read_ndjson(JSON_FILE), 20))

prices  = list(map(lambda tx: tx["price"], sample))
totals  = list(map(lambda tx: round(tx["price"] * tx["volume"], 2), sample))
high    = list(filter(lambda tx: tx["price"] > 100, sample))

print(f"map → ціни перших 5: {prices[:5]}")
print(f"map → обсяги перших 3: {totals[:3]}")
print(f"filter → угоди > 100₴: {len(high)} з {len(sample)}")

prev_prices = prices[:-1]
curr_prices = prices[1:]
changes = list(map(lambda pair: round((pair[1] - pair[0]) / pair[0] * 100, 3),
                   zip(prev_prices, curr_prices)))
print(f"zip → зміна ціни (%): {changes[:5]}")

for i, tx in enumerate(islice(read_ndjson(JSON_FILE), 3), start=1):
    print(f"enumerate → [{i}] {tx['company']}: ₴{tx['price']}")

above_200 = any(tx["price"] > 200 for tx in sample)
all_valid  = all(tx["volume"] > 0 for tx in sample)
print(f"any(price > 200): {above_200}  |  all(volume > 0): {all_valid}")

## 11. Системне Мислення: Дані Не Зберігаються — Вони Течуть

---

Ось повний конвеєр від диску до результату. Зверніть увагу:  
жоден проміжний список не існує в пам'яті в жодний момент часу.

```
Диск (файл 5 ГБ)
   └── read_ndjson()         ← генератор читає по 1 рядку
        └── filter_by_company()  ← відкидає 4/5 рядків
             └── filter_by_price()  ← відкидає угоди поза діапазоном
                  └── enrich_with_total()  ← додає поле total
                       └── aggregate()  ← збирає статистику
                            └── Результат (один dict)
```

**RAM у будь-який момент часу:** лише 1 транзакція + поточний стан агрегатора.

In [ ]:
def build_pipeline(path: Path, company: str, min_price: float, max_price: float):
    return aggregate(
        enrich_with_total(
            filter_by_price(
                filter_by_company(
                    read_ndjson(path),
                    company=company,
                ),
                min_price=min_price,
                max_price=max_price,
            )
        )
    )


for company in COMPANIES:
    cfg  = COMPANIES[company]
    lo   = cfg["base_price"] * 0.95
    hi   = cfg["base_price"] * 1.05
    res  = build_pipeline(JSON_FILE, company, lo, hi)
    print(f"{company:<14}  {res['count']:>5} угод у ±5% діапазоні  VWAP={res['vwap']}₴")

## 12. Реальний Світ: Де Ще Застосовується Цей Підхід

---

### Pandas: `chunksize`
```python
for chunk in pd.read_csv('transactions.csv', chunksize=10_000):
    process(chunk)  # Pandas читає по 10k рядків, як ітератор
```

### FastAPI: `StreamingResponse`
```python
async def generate_csv():
    for tx in read_ndjson('transactions.ndjson'):
        yield json.dumps(tx) + '\n'   # Генератор → HTTP потік

return StreamingResponse(generate_csv(), media_type='application/x-ndjson')
```

### Dask: Ледачий DataFrame
```python
import dask.dataframe as dd
df = dd.read_csv('*.csv')   # Нічого не завантажено!
result = df.groupby('company')['price'].mean().compute()  # Лише тут виконується
```

### Apache Kafka
```python
consumer = KafkaConsumer('transactions')
for message in consumer:   # Нескінченний ітератор подій
    process(message.value)
```

Усі ці системи — лише різні реалізації одного й того самого патерну: **Iterator Protocol**.

## 13. Фінальний Інсайт: Три Рівні Розуміння

---

```
┌─────────────────────────────────────────────────────────┐
│  ITERATOR  = стандарт доступу до даних                  │
│             (будь-який об'єкт → for loop)               │
├─────────────────────────────────────────────────────────┤
│  GENERATOR = фабрика значень                            │
│             (yield → пауза → стан збережено)            │
├─────────────────────────────────────────────────────────┤
│  STREAMING = модель обчислень                           │
│             (дані не зберігаються — вони течуть)         │
└─────────────────────────────────────────────────────────┘
```

| Рівень | Інструмент | Застосування |
|--------|------------|-------------|
| Новачок | `list` | Маленькі дані, багаторазовий доступ |
| Розробник | `generator` | Великі дані, одноразовий прохід |
| Архітектор | Pipeline | Системи реального часу, ETL, стрімінг |

> **Наступний крок:** запустіть `app_transactions.py` — Streamlit-застосунок,  
> який візуалізує цей генераторний конвеєр у реальному часі.

 python -m streamlit run module_2/lessons/lesson_19_iterators_generators/app_transactions.py
